# Random Forest — Detección temprana de alertas (v2)

**Enfoque inspirado en Taggart (2026):**
https://pub.towardsai.net/can-you-predict-a-subway-delay-before-transit-officials-announce-it-ec16cab64149
Se demostró que los retrasos del metro pueden anticiparse antes de los anuncios oficiales de la MTA. El enfoque agrega datos de trenes en ventanas de 30 minutos por parada, extrayendo features estadísticas que capturan la degradación del servicio.Nuestro modelo sigue esta misma metodología, adaptándola a toda la red del metro de Nueva York durante 2025.
Trabajamos con ventanas de 30 minutos

## 0) Setup

A rasgos generales, el Random Forest es un modelo de clasificación que construye múltiples árboles de decisión de forma independiente y combina 
sus predicciones por votación . Cada árbol se entrena sobre una muestra aleatoria del dataset y considera solo un subconjunto aleatorio de features en cada split, lo que reduce el sobreajuste y mejora la generalización. En nuestro caso, el modelo recibe las características operativas de una ventana de 30 minutos y predice si habrá una alerta oficial de la MTA en los próximos 30 minutos

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, precision_recall_curve
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook')
plt.rcParams['figure.figsize'] = (14, 5)

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet

TARGET_COL   = 'alert_in_next_30m_max'
RANDOM_STATE = 42

print('✓ Setup completado')

## 1) Carga de datos desde MinIO

Dataset agregado en ventanas de 30 minutos por parada — todas las líneas, enero–diciembre 2025.

## Features utilizadas
### Identificadores
| Feature | Descripción |
|---|---|
| `route_id` | Línea del metro (1, 2, A, C...) |
| `direction` | Dirección del tren (N/S) |
### Retraso actual
| Feature | Descripción |
|---|---|
| `delay_seconds_mean` | Retraso medio de los trenes en la ventana de 30 min |
| `delay_seconds_max` | Retraso máximo en la ventana de 30 min |
| `lagged_delay_1_mean` | Retraso medio de la parada anterior dentro del mismo viaje |
| `lagged_delay_2_mean` | Retraso medio de hace dos paradas dentro del mismo viaje |
| `route_rolling_delay_mean` | Media móvil de retrasos recientes en toda la línea |
| `route_rolling_delay_max` | Máximo móvil de retrasos recientes en toda la línea |
### Servicio
| Feature | Descripción |
|---|---|
| `actual_headway_seconds_mean` | Intervalo medio entre trenes en esa parada |
| `match_key_nunique` | Número de trenes distintos en la ventana de 30 min |
| `stops_to_end_mean` | Paradas que quedan hasta el final del trayecto |
| `scheduled_time_to_end_mean` | Tiempo programado hasta el final del trayecto (segundos) |
| `is_unscheduled_max` | Si hubo algún tren no programado en la ventana |
| `num_updates_sum` | Total de actualizaciones de posición enviadas en la ventana |
### Temporales
| Feature | Descripción |
|---|---|
| `hour_sin_first` | Hora del día codificada como seno (cíclica) |
| `hour_cos_first` | Hora del día codificada como coseno (cíclica) |
| `dow_first` | Día de la semana (0=lunes) |
| `is_weekend_max` | Si es fin de semana |
### Clima
| Feature | Descripción |
|---|---|
| `temp_extreme_max` | Si hubo temperatura extrema (frío o calor intenso) |
### Historial de alertas
| Feature | Descripción |
|---|---|
| `seconds_since_last_alert_mean` | Segundos transcurridos desde la última alerta oficial en esa línea

In [ ]:
access_key = os.getenv('MINIO_ACCESS_KEY')
secret_key = os.getenv('MINIO_SECRET_KEY')
if not access_key or not secret_key:
    raise ValueError('MINIO_ACCESS_KEY y MINIO_SECRET_KEY deben estar definidas')

MINIO_PATH = 'grupo5/aggregations/DataFrameGroupedByMin=30.parquet'

print('Cargando dataset agregado 30 min desde MinIO...')
df_raw = download_df_parquet(access_key, secret_key, MINIO_PATH)


COLS_NECESARIAS = [
    'merge_time', 'route_id', 'direction',
    'alert_in_next_30m_max',
    # Retraso
    'delay_seconds_mean', 'delay_seconds_max',
    'lagged_delay_1_mean', 'lagged_delay_2_mean',
    'route_rolling_delay_mean', 'route_rolling_delay_max',
    # Headway
    'actual_headway_seconds_mean',
    # Trenes en la ventana
    'match_key_nunique',
    # Temporales
    'hour_sin_first', 'hour_cos_first', 'dow_first', 'is_weekend_max',
    'scheduled_time_to_end_mean', 'stops_to_end_mean',
    # Servicio
    'is_unscheduled_max', 'num_updates_sum',
    # Clima
    'temp_extreme_max',
    # Historial alertas
    'seconds_since_last_alert_mean',
]

# Filtrar columnas existentes y seleccionar solo esas
COLS_NECESARIAS = [c for c in COLS_NECESARIAS if c in df_raw.columns]
df = df_raw[COLS_NECESARIAS].copy()
del df_raw  # liberar los 80 columnas originales

# Convertir float64 → float32 para reducir memoria a la mitad
for col in df.select_dtypes(include='float64').columns:
    df[col] = df[col].astype('float32')

mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f'\n✓ Dataset cargado y optimizado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'  Memoria en RAM: {mem_gb:.2f} GB')
print(f'  Periodo: {df["merge_time"].min()} → {df["merge_time"].max()}')
print(f'  Positivos: {df["alert_in_next_30m_max"].sum():,} ({df["alert_in_next_30m_max"].mean()*100:.1f}%)')
df.head(3)

In [ ]:
df

## 2) Selección de features

Usamos 19 features limpias sin leakage. Las columnas `target_delay_*`, `delta_delay_*`
y `seconds_to_next_alert_mean` se excluyen por ser información del futuro.

`match_key_nunique` es el número de trenes distintos en la ventana de 30 min —
si hay menos trenes de lo normal, es señal directa de problema de servicio (similar
a los dwell time stats de Taggart).

In [ ]:
FEATURE_COLS = [
    'delay_seconds_mean', 'delay_seconds_max',
    'lagged_delay_1_mean', 'lagged_delay_2_mean',
    'route_rolling_delay_mean', 'route_rolling_delay_max',
    'actual_headway_seconds_mean',
    'match_key_nunique',
    'hour_sin_first', 'hour_cos_first', 'dow_first', 'is_weekend_max',
    'scheduled_time_to_end_mean', 'stops_to_end_mean',
    'is_unscheduled_max', 'num_updates_sum',
    'temp_extreme_max',
    'seconds_since_last_alert_mean',
]

CAT_COLS = ['direction', 'route_id']

df['merge_time'] = pd.to_datetime(df['merge_time'])
df = df.sort_values('merge_time').reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)


encoders = {}
for col in CAT_COLS:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str).fillna('UNKNOWN'))
        encoders[col] = le
        print(f'  Encoded "{col}": {len(le.classes_)} categorías')

# Añadir categóricas a features
FEATURE_COLS = FEATURE_COLS + [c for c in CAT_COLS if c in df.columns]

# Imputar NaN
df['seconds_since_last_alert_mean'] = df['seconds_since_last_alert_mean'].fillna(999999)
cols_mediana = [c for c in FEATURE_COLS if c != 'seconds_since_last_alert_mean']
df[cols_mediana] = df[cols_mediana].fillna(df[cols_mediana].median(numeric_only=True))

print(f'\nDataset listo: {len(df):,} filas')
print(f'  Features totales: {len(FEATURE_COLS)}')
print(f'  Positivos: {df[TARGET_COL].sum():,} ({df[TARGET_COL].mean()*100:.1f}%)')
print(f'  Negativos: {(df[TARGET_COL]==0).sum():,} ({(df[TARGET_COL]==0).mean()*100:.1f}%)')

## 3) Split temporal 70 / 15 / 15

Split temporal obligatorio para series temporales: nunca aleatorio.
Train = primeros 70% del año, Val = siguientes 15%, Test = últimos 15%.

In [ ]:
n = len(df)
i_train = int(n * 0.70)
i_val   = int(n * 0.85)

df_train = df.iloc[:i_train]
df_val   = df.iloc[i_train:i_val]
df_test  = df.iloc[i_val:]

X_train, y_train = df_train[FEATURE_COLS], df_train[TARGET_COL]
X_val,   y_val   = df_val[FEATURE_COLS],   df_val[TARGET_COL]
X_test,  y_test  = df_test[FEATURE_COLS],  df_test[TARGET_COL]

for name, subset, y in [('Train', df_train, y_train),
                         ('Val',   df_val,   y_val),
                         ('Test',  df_test,  y_test)]:
    pos = y.sum()
    print(f'{name:5s}: {len(subset):>9,} obs | pre-alerta: {pos:>6,} ({pos/len(subset)*100:.2f}%) '
          f'| {subset["merge_time"].min().date()} → {subset["merge_time"].max().date()}')

## 4) Entrenamiento del Random Forest

## Parámetros del RandomForestClassifier

| Parámetro | Valor usado | Descripción |
|---|---|---|
| `n_estimators` | 100 | Número de árboles del bosque. Más árboles = más estable pero más lento |
| `max_depth` | 10 | Profundidad máxima de cada árbol. Limita la complejidad y evita sobreajuste |
| `min_samples_leaf` | 50 | Mínimo de muestras que debe tener una hoja para existir. Evita reglas para muy pocos casos |
| `class_weight` | {0:1, 1:3} | Penalización por clase. Le dice al modelo que un fallo en clase 1 (alerta) pesa 3 veces más que en clase 0 |
| `n_jobs` | -1 | Número de núcleos del procesador usados. -1 significa todos los disponibles |
| `random_state` | 42 | Semilla aleatoria para que los resultados sean reproducibles |
| `max_features` | sqrt (default) | Número de features consideradas en cada split. Por defecto la raíz cuadrada del total |

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=50,
    class_weight={0: 1, 1: 3},
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)

print('Entrenando Random Forest (dataset agregado 30 min)...')
t0 = time.time()
rf.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'✓ Entrenamiento completado en {elapsed/60:.1f} minutos')

## 5) Búsqueda de hiperparámetros — RandomizedSearchCV



**Problema con 18M filas:** CV normal sobre todo el train tardaría horas.  
**Solución:** usamos un **30% del train** para la búsqueda


luego **reentrenamos el mejor modelo sobre el train completo** para el modelo final.

- `TimeSeriesSplit(n_splits=3)`: validación cruzada respetando el orden temporal (no hay data leakage)
- 
- `n_iter=15`: probamos 15 combinaciones aleatorias del espacio de hiperparámetros
- 
- `scoring='f1'`: optimizamos F1 porque tenemos clases desbalanceadas
- 
- Parámetros explorados: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, `class_weight`

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

n_sample = int(len(X_train) * 0.30)
X_train_sample = X_train.iloc[-n_sample:]   # últimos 30% = los más recientes
y_train_sample = y_train.iloc[-n_sample:]

print(f'Muestra para RandomizedSearchCV: {len(X_train_sample):,} filas '
      f'({len(X_train_sample)/len(X_train)*100:.0f}% del train)')
print(f'Positivos en muestra: {y_train_sample.sum():,} ({y_train_sample.mean()*100:.1f}%)')

param_dist = {
    'n_estimators':      [50, 100, 200],         
    'max_depth':         [8, 10, 12, 15, None],   
    'min_samples_leaf':  [20, 50, 100, 200],      
    'max_features':      ['sqrt', 'log2', 0.3],   
    'class_weight':      [                        
        {0: 1, 1: 2},
        {0: 1, 1: 3},  
        {0: 1, 1: 5},
        'balanced',
    ],
}

tscv = TimeSeriesSplit(n_splits=3)

rf_search = RandomForestClassifier(
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)

search = RandomizedSearchCV(
    estimator=rf_search,
    param_distributions=param_dist,
    n_iter=15,           # 15 combinaciones aleatorias
    cv=tscv,
    scoring='f1',        # optimizamos F1 (clases desbalanceadas)
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=2,
    refit=True           # refitea automáticamente con los mejores params
)

print('\nEjecutando RandomizedSearchCV (puede tardar ~15-25 min)...')
t0 = time.time()
search.fit(X_train_sample, y_train_sample)
elapsed = time.time() - t0

print(f'\n✓ Búsqueda completada en {elapsed/60:.1f} minutos')
print(f'\n=== Mejores hiperparámetros encontrados ===')
for param, val in search.best_params_.items():
    print(f'  {param}: {val}')
print(f'\nMejor F1 en CV (muestra 30%): {search.best_score_:.4f}')

df_cv_results = pd.DataFrame(search.cv_results_)[
    ['rank_test_score', 'mean_test_score', 'std_test_score', 'params']
].sort_values('rank_test_score')
print('\nTop 5 combinaciones:')
display(df_cv_results.head(5))

### 5b) Reentrenamiento con mejores parámetros sobre el train completo

La búsqueda se hizo sobre el 30% del train para ser manejable.  
Ahora tomamos los mejores parámetros y **reentrenamos sobre los 12.9M filas completos**,
que es lo que usará el modelo final para competir contra XGBoost y Logistic Regression.

In [ ]:
# ── Reentrenar con mejores params sobre el train completo ──────────────────────
best_params = search.best_params_
print('Reentrenando con mejores parámetros sobre TRAIN COMPLETO...')
print(f'  Train: {len(X_train):,} filas')
print(f'  Params: {best_params}')

rf_opt = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)
def evaluar_modelo(nombre, model, X, y):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        "Modelo":    nombre,
        "Accuracy":  round(accuracy_score(y, y_pred), 4),
        "Precision": round(precision_score(y, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y, y_pred, zero_division=0), 4),
        "F1":        round(f1_score(y, y_pred, zero_division=0), 4),
        "AUC-ROC":   round(roc_auc_score(y, y_proba), 4),
    }
t0 = time.time()
rf_opt.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'✓ Reentrenamiento completado en {elapsed/60:.1f} minutos')

res_base_test = evaluar_modelo('RF base (n=100, depth=10)', rf,     X_test, y_test)
res_opt_val   = evaluar_modelo('RF optimizado — Validación',  rf_opt, X_val,  y_val)
res_opt_test  = evaluar_modelo('RF optimizado — Test',        rf_opt, X_test, y_test)

df_comparacion = pd.DataFrame([res_base_test, res_opt_val, res_opt_test]).set_index('Modelo')
print('\n=== Comparación: Base vs RandomizedSearchCV ===')
display(df_comparacion)

delta_f1  = res_opt_test['F1']  - res_base_test['F1']
delta_auc = res_opt_test['AUC-ROC'] - res_base_test['AUC-ROC']
print(f'\nMejora en Test:')
print(f'  ΔF1:      {delta_f1:+.4f}')
print(f'  ΔAUC-ROC: {delta_auc:+.4f}')

rf_final = rf_opt
print('\n✓ rf_final = modelo optimizado listo para la sección de evaluación')

## 5) Evaluación

In [ ]:
def evaluar_modelo(nombre, model, X, y):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        'Modelo':    nombre,
        'Precision': round(precision_score(y, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y, y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y, y_proba), 4),
    }

res_val  = evaluar_modelo('RF v2 — Validación', rf, X_val,  y_val)
res_test = evaluar_modelo('RF v2 — Test',       rf, X_test, y_test)

df_res = pd.DataFrame([res_val, res_test]).set_index('Modelo')
print('=== Resultados Random Forest v2 (dataset 30 min) ===')
display(df_res)

print('\n=== Reporte detallado (Test) ===')
y_pred_test = rf.predict(X_test)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Pre-alerta']))

## 6) Búsqueda de hiperparámetros — Optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, TimeSeriesSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Misma muestra del 30% que RandomizedSearchCV ─────────────────────────────
n_sample = int(len(X_train) * 0.30)
X_train_sample = X_train.iloc[-n_sample:]
y_train_sample = y_train.iloc[-n_sample:]

tscv = TimeSeriesSplit(n_splits=3)

# ── Función objetivo que Optuna intenta maximizar ─────────────────────────────
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_categorical('n_estimators', [50, 100, 200]),
        'max_depth':        trial.suggest_categorical('max_depth', [8, 10, 12, 15]),
        'min_samples_leaf': trial.suggest_categorical('min_samples_leaf', [20, 50, 100, 200]),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3]),
        'class_weight':     trial.suggest_categorical('class_weight', ['balanced', 'cw_1_2', 'cw_1_3', 'cw_1_5']),
    }

    # Convertir strings a dicts para class_weight
    cw_map = {
        'cw_1_2': {0: 1, 1: 2},
        'cw_1_3': {0: 1, 1: 3},
        'cw_1_5': {0: 1, 1: 5},
        'balanced': 'balanced',
    }
    params['class_weight'] = cw_map[params['class_weight']]

    rf_trial = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    scores = cross_val_score(
        rf_trial, X_train_sample, y_train_sample,
        cv=tscv, scoring='f1', n_jobs=-1
    )
    return scores.mean()

# ── Ejecutar la búsqueda ──────────────────────────────────────────────────────
print('Ejecutando Optuna (20 trials)...')
t0 = time.time()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

elapsed = time.time() - t0
print(f'\n✓ Optuna completado en {elapsed/60:.1f} minutos')
print(f'\n=== Mejores hiperparámetros (Optuna) ===')
for param, val in study.best_params.items():
    print(f'  {param}: {val}')
print(f'\nMejor F1 en CV (Optuna): {study.best_value:.4f}')
print(f'Mejor F1 en CV (RandomizedSearch): {search.best_score_:.4f}')

# ── Reentrenar con mejores params de Optuna sobre train completo ──────────────
cw_map = {
    'cw_1_2': {0: 1, 1: 2},
    'cw_1_3': {0: 1, 1: 3},
    'cw_1_5': {0: 1, 1: 5},
    'balanced': 'balanced',
}

best_params_optuna = study.best_params.copy()
best_params_optuna['class_weight'] = cw_map[best_params_optuna['class_weight']]

print('\nReentrenando con mejores params de Optuna sobre TRAIN COMPLETO...')
rf_optuna = RandomForestClassifier(
    **best_params_optuna,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
t0 = time.time()
rf_optuna.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'✓ Reentrenamiento completado en {elapsed/60:.1f} minutos')

# ── Comparación final: Base vs RandomizedSearchCV vs Optuna ──────────────────
res_base     = evaluar_modelo('RF base',             rf,        X_test, y_test)
res_rscv     = evaluar_modelo('RF RandomizedSearchCV', rf_opt,  X_test, y_test)
res_optuna   = evaluar_modelo('RF Optuna',           rf_optuna, X_test, y_test)

df_final = pd.DataFrame([res_base, res_rscv, res_optuna]).set_index('Modelo')
print('\n=== Comparación final en Test ===')
display(df_final)

## 6) Curva Precision-Recall y threshold óptimo
El threshold es el punto de corte que decides tú para convertir la probabilidad en una predicción binaria (0 o 1).


In [ ]:
y_proba_test = rf.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_test)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Threshold óptimo (máx F1): {best_threshold:.3f}')
print(f'  Precision: {precisions[best_idx]:.4f}')
print(f'  Recall:    {recalls[best_idx]:.4f}')
print(f'  F1:        {f1_scores[best_idx]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recalls, precisions, color='steelblue', lw=2)
axes[0].axvline(recalls[best_idx], color='tomato', linestyle='--',
                label=f'Threshold óptimo ({best_threshold:.2f})')
axes[0].scatter(recalls[best_idx], precisions[best_idx], color='tomato', zorder=5)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Curva Precision-Recall — RF v2 (30 min)')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(thresholds, f1_scores[:-1], color='green', lw=2, label='F1')
axes[1].plot(thresholds, precisions[:-1], color='steelblue', lw=1.5, linestyle='--', label='Precision')
axes[1].plot(thresholds, recalls[:-1], color='orange', lw=1.5, linestyle='--', label='Recall')
axes[1].axvline(best_threshold, color='tomato', linestyle='--', label=f'Mejor F1 (th={best_threshold:.2f})')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('F1 / Precision / Recall vs Threshold')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

y_pred_opt = (y_proba_test >= best_threshold).astype(int)
print(f'\n=== Resultados con threshold óptimo ({best_threshold:.3f}) ===')
print(classification_report(y_test, y_pred_opt, target_names=['Normal', 'Pre-alerta']))

## 7) Importancia de variables

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['tomato' if i < 5 else 'steelblue' for i in range(len(importances))]
ax.barh(importances.index[::-1], importances.values[::-1], color=colors[::-1])
ax.set_title('Importancia de variables — RF v2 (dataset 30 min)')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout(); plt.show()

print('Top 5 variables más predictivas:')
for i, (feat, imp) in enumerate(importances.head(5).items(), 1):
    print(f'  {i}. {feat}: {imp:.4f} ({imp*100:.1f}%)')

## 8) Matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opt,
    display_labels=['Normal', 'Pre-alerta'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Matriz de confusión — RF v2 (threshold={best_threshold:.2f})')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_opt).ravel()
print(f'Verdaderos negativos:  {tn:,}')
print(f'Falsos positivos:      {fp:,}  ← alarmas falsas')
print(f'Falsos negativos:      {fn:,}  ← alertas no detectadas')
print(f'Verdaderos positivos:  {tp:,}  ← alertas detectadas correctamente')

---
# Segunda aproximación — Modelo especializado: Línea 1, año completo 2025

Cambios respecto a la primera aproximación:
- **Datos**: `grupo5/aggregations/lines/line=1/dataset_final.parquet` (enero–diciembre 2025)
- **`route_id` eliminada de features**: valor constante en un modelo de una sola línea
- El resto del pipeline es idéntico para que la comparación sea directa

## A) Carga de datos — Línea 1, año completo

In [ ]:
LINEA = "2"
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")
MINIO_PATH_L1 = f"grupo5/aggregations/lines/line={LINEA}/dataset_final.parquet"

print(f"Cargando dataset línea {LINEA} desde MinIO...")
df2 = download_df_parquet(access_key, secret_key, MINIO_PATH_L1)

print(f"\n✓ Dataset cargado: {df2.shape[0]:,} filas × {df2.shape[1]} columnas")
print(f"  Periodo: {df2['date'].min()} → {df2['date'].max()}")
print(f"  Positivos: {df2[TARGET_COL].sum():,} ({df2[TARGET_COL].mean()*100:.1f}%)")
df2.head(3)


## B) Features, encoding y filtro de negativos

In [ ]:

df2['merge_time'] = pd.to_datetime(df2['merge_time'])

df2 = df2.sort_values('merge_time').reset_index(drop=True)
roll_linea = df2.set_index('merge_time')['actual_headway_seconds'].rolling('30min', min_periods=1)
df2['headway_linea_mean'] = roll_linea.mean().values   # media: ¿cuánto esperan los trenes?
df2['headway_linea_max']  = roll_linea.max().values    # máximo: ¿hay algún tren muy retrasado?
df2['headway_linea_std']  = roll_linea.std().values    # std: ¿los trenes llegan de forma irregular?
df2['headway_ratio']      = df2['actual_headway_seconds'] / (df2['headway_linea_mean'] + 1)
# ratio > 1 → este tren llega más tarde que la media reciente → señal de problema
print("✓ Estadísticos de headway a nivel de línea calculados (estilo Taggart)")

# ── Features de ventana deslizante por parada ─────────────────────────────────
df2 = df2.sort_values(['stop_id', 'merge_time'])
ventana = 4  # ~30 minutos

df2['headway_rolling_std'] = df2.groupby('stop_id', observed=True)['actual_headway_seconds']\
    .transform(lambda x: x.rolling(ventana, min_periods=1).std())
df2['delay_rolling_max'] = df2.groupby('stop_id', observed=True)['delay_seconds']\
    .transform(lambda x: x.rolling(ventana, min_periods=1).max())
df2['delay_aceleracion'] = df2.groupby('stop_id', observed=True)['delay_seconds']\
    .transform(lambda x: x.diff())
print("✓ Features de ventana deslizante por parada calculadas")

FEATURE_COLS_L1 = [
    'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
    'route_rolling_delay', 'actual_headway_seconds',
    'is_unscheduled', 'num_updates', 'scheduled_time_to_end',
    'stops_to_end', 'direction',
    'hour_sin', 'hour_cos', 'dow', 'is_weekend',
    'temp_extreme',
    'seconds_since_last_alert',
    'headway_rolling_std', 'delay_rolling_max', 'delay_aceleracion',
    # Nuevas: headway a nivel de línea completa (estilo Taggart)
    'headway_linea_mean',
    'headway_linea_max',
    'headway_linea_std',
    'headway_ratio',
]

CAT_COLS_L1 = ['direction', 'tipo_referente']

df2 = df2.sort_values('merge_time').reset_index(drop=True)
df2 = df2.dropna(subset=[TARGET_COL]).copy()
df2[TARGET_COL] = df2[TARGET_COL].astype(int)

encoders_l1 = {}
for col in CAT_COLS_L1:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col].astype(str).fillna('UNKNOWN'))
    encoders_l1[col] = le
    print(f"  Encoded '{col}': {len(le.classes_)} categorías")

# Filtro de negativos ambiguos (Taggart 2026)
mask_pos = df2[TARGET_COL] == 1
mask_neg = df2['seconds_to_next_alert'].isna() | (df2['seconds_to_next_alert'] > 3600)
df2 = df2[mask_pos | mask_neg].copy().reset_index(drop=True)

# seconds_since_last_alert: NaN = nunca hubo alerta → valor alto
df2['seconds_since_last_alert'] = df2['seconds_since_last_alert'].fillna(999999)

# Resto de NaN con la mediana
cols_mediana = [c for c in FEATURE_COLS_L1 if c != 'seconds_since_last_alert']
df2[cols_mediana] = df2[cols_mediana].fillna(df2[cols_mediana].median(numeric_only=True))

print(f"\nDataset listo: {len(df2):,} filas")
print(f"  Features totales: {len(FEATURE_COLS_L1)}")
print(f"  Positivos: {df2[TARGET_COL].sum():,} ({df2[TARGET_COL].mean()*100:.1f}%)")
print(f"  Negativos: {(df2[TARGET_COL]==0).sum():,} ({(df2[TARGET_COL]==0).mean()*100:.1f}%)")

## C) Split temporal 70 / 15 / 15

In [ ]:
n2 = len(df2)
i_train2 = int(n2 * 0.70)
i_val2   = int(n2 * 0.85)

df2_train = df2.iloc[:i_train2]
df2_val   = df2.iloc[i_train2:i_val2]
df2_test  = df2.iloc[i_val2:]

X2_train, y2_train = df2_train[FEATURE_COLS_L1], df2_train[TARGET_COL]
X2_val,   y2_val   = df2_val[FEATURE_COLS_L1],   df2_val[TARGET_COL]
X2_test,  y2_test  = df2_test[FEATURE_COLS_L1],  df2_test[TARGET_COL]

for name, subset, y in [("Train", df2_train, y2_train),
                         ("Val",   df2_val,   y2_val),
                         ("Test",  df2_test,  y2_test)]:
    pos = y.sum()
    print(f"{name:5s}: {len(subset):>9,} obs | pre-alerta: {pos:>6,} ({pos/len(subset)*100:.2f}%) "
          f"| {subset['merge_time'].min().date()} → {subset['merge_time'].max().date()}")

## D) Entrenamiento — Línea 1

In [ ]:
import time

rf_l1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=50,
    class_weight={0: 1, 1: 3},  
                                  
                                 
                                  
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)

print("Entrenando Random Forest — Línea 1 (class_weight={0:1, 1:3})...")
t0 = time.time()
rf_l1.fit(X2_train, y2_train)
elapsed = time.time() - t0
print(f"✓ Entrenamiento completado en {elapsed/60:.1f} minutos")

## E) Evaluación y comparación con primera aproximación

In [ ]:
def evaluar_modelo(nombre, model, X, y):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        "Modelo":    nombre,
        "Accuracy":  round(accuracy_score(y, y_pred), 4),
        "Precision": round(precision_score(y, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y, y_pred, zero_division=0), 4),
        "F1":        round(f1_score(y, y_pred, zero_division=0), 4),
        "AUC-ROC":   round(roc_auc_score(y, y_proba), 4),
    }
res_l1_val  = evaluar_modelo("RF L1 — Validación", rf_l1, X2_val,  y2_val)
res_l1_test = evaluar_modelo("RF L1 — Test",       rf_l1, X2_test, y2_test)

# res_test puede no estar definido si se ejecuta solo la Parte 2
try:
    fila_general = {
        "Modelo":    "RF General — Feb 2025 (Test)",
        "Datos":     "Feb 2025, todas las líneas",
        "Precision": res_test["Precision"],
        "Recall":    res_test["Recall"],
        "F1":        res_test["F1"],
        "AUC-ROC":   res_test["AUC-ROC"],
    }
except NameError:
    fila_general = {
        "Modelo":    "RF General — Feb 2025 (Test)",
        "Datos":     "No disponible (ejecuta Parte 1 primero)",
        "Precision": None, "Recall": None, "F1": None, "AUC-ROC": None,
    }

comparacion = pd.DataFrame([
    fila_general,
    {
        "Modelo":    "RF Línea 1 — Año completo (Test)",
        "Datos":     "Ene–Dic 2025, solo línea 1",
        "Precision": res_l1_test["Precision"],
        "Recall":    res_l1_test["Recall"],
        "F1":        res_l1_test["F1"],
        "AUC-ROC":   res_l1_test["AUC-ROC"],
    },
]).set_index("Modelo")

print("=== Comparación de aproximaciones ===")
display(comparacion)

print("\n=== Reporte detallado — Línea 1 (Test) ===")
THRESHOLD = 0.392
y2_proba_test = rf_l1.predict_proba(X2_test)[:, 1]
y2_pred_test = (y2_proba_test >= THRESHOLD).astype(int)
print(f"(Usando threshold = {THRESHOLD})")
print(classification_report(y2_test, y2_pred_test, target_names=["Normal", "Pre-alerta"]))

In [ ]:
importances_l1 = pd.Series(rf_l1.feature_importances_, index=FEATURE_COLS_L1)
importances_l1 = importances_l1.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["tomato" if i < 5 else "steelblue" for i in range(len(importances_l1))]
ax.barh(importances_l1.index[::-1], importances_l1.values[::-1], color=colors[::-1])
ax.set_title("Importancia de variables — RF Línea 1 (año completo)")
ax.set_xlabel("Importancia (Gini)")
plt.tight_layout()
plt.show()

print("Top 5 variables más predictivas:")
for i, (feat, imp) in enumerate(importances_l1.head(5).items(), 1):
    print(f"  {i}. {feat}: {imp:.4f} ({imp*100:.1f}%)")

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np

# Probabilidades del modelo en test
y2_proba_test = rf_l1.predict_proba(X2_test)[:, 1]

# Curva Precision-Recall
precisions, recalls, thresholds = precision_recall_curve(y2_test, y2_proba_test)

# F1 para cada threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Threshold óptimo (máx F1): {best_threshold:.3f}")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print(f"  F1:        {f1_scores[best_idx]:.4f}")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Curva PR —
axes[0].plot(recalls, precisions, color='steelblue', lw=2)
axes[0].axvline(recalls[best_idx], color='tomato', linestyle='--', label=f'Threshold óptimo ({best_threshold:.2f})')
axes[0].scatter(recalls[best_idx], precisions[best_idx], color='tomato', zorder=5)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Curva Precision-Recall — RF Línea 1")
axes[0].legend()
axes[0].grid(True)


axes[1].plot(thresholds, f1_scores[:-1], color='green', lw=2)
axes[1].plot(thresholds, precisions[:-1], color='steelblue', lw=1.5, linestyle='--', label='Precision')
axes[1].plot(thresholds, recalls[:-1], color='orange', lw=1.5, linestyle='--', label='Recall')
axes[1].axvline(best_threshold, color='tomato', linestyle='--', label=f'Mejor F1 (th={best_threshold:.2f})')
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("F1 / Precision / Recall vs Threshold")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Evaluación con el threshold óptimo
y2_pred_opt = (y2_proba_test >= best_threshold).astype(int)
print("\n=== Resultados con threshold óptimo ===")
print(classification_report(y2_test, y2_pred_opt, target_names=["Normal", "Pre-alerta"]))